# 07_03E — Graph Node2Vec Features + Modelagem Early-Stage

**Objetivo:** criar embeddings de processos usando estrutura de grafo com Node2Vec e testar os mesmos modelos dos notebooks anteriores.

Semântica do target:

- `0 = banco ganhou`
- `1 = banco perdeu`

Este notebook constrói um grafo heterogêneo conectando `processo -> entidades jurídicas` e transforma cada processo em um vetor numérico (`graph_emb_*`). Esses embeddings são adicionados às features históricas/tabulares/textuais já existentes.

> Atenção contra leakage: o grafo não usa `target`, `Motivo_Encerramento`, resultado, sentença, acordo, valor de condenação ou qualquer campo pós-desfecho. Para validação temporal, o grafo de cada fold é construído usando apenas atributos conhecidos no momento da predição.

In [ ]:
# ============================================================
# 0. Imports e configurações gerais
# ============================================================

import os
import re
import math
import json
import time
import random
import warnings
from pathlib import Path
from collections import defaultdict, Counter

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception as e:
    HAS_XGBOOST = False
    print("XGBoost não disponível:", e)

try:
    from lightgbm import LGBMClassifier
    HAS_LIGHTGBM = True
except Exception as e:
    HAS_LIGHTGBM = False
    print("LightGBM não disponível:", e)

try:
    from gensim.models import Word2Vec
    HAS_GENSIM = True
except Exception as e:
    HAS_GENSIM = False
    print("Gensim não disponível. Será usado fallback por SVD de matriz processo-entidade:", e)

try:
    from scipy import sparse
    from sklearn.decomposition import TruncatedSVD
    HAS_SCIPY = True
except Exception as e:
    HAS_SCIPY = False
    print("Scipy/SVD não disponível:", e)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## 1. Parâmetros do notebook

Ajuste os caminhos conforme a sua estrutura local. O notebook espera usar como entrada os arquivos gerados pelo `07_01_prepare_early_stage_abt.ipynb`.

In [ ]:
# ============================================================
# 1. Configuração de caminhos e parâmetros
# ============================================================

ROOT_DIR = Path("..").resolve() if Path("..").exists() else Path(".").resolve()

DATA_DIR = ROOT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = ROOT_DIR / "outputs" / "reports" / "modelagem_early_stage"
MODELS_DIR = ROOT_DIR / "outputs" / "models" / "modelagem_early_stage"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DEV_PATH = PROCESSED_DIR / "abt_early_stage_v1_dev.parquet"
HOLDOUT_PATH = PROCESSED_DIR / "abt_early_stage_v1_holdout.parquet"
FEATURE_LIST_PATH = PROCESSED_DIR / "feature_list_early_stage_v1.txt"

TARGET_COL = "target"
ID_COL_CANDIDATES = ["Numerodistribuicao", "numero_processo", "processo_id", "Identificador", "id_processo"]
DATE_COL_CANDIDATES = ["Data_Distribuicao", "data_distribuicao", "data_inicio_processo", "Dt_Distribuicao", "data_ref"]

RUN_MODE = "fast"  # "fast" ou "full"

if RUN_MODE == "fast":
    N2V_DIM = 32
    N2V_WALK_LENGTH = 16
    N2V_NUM_WALKS = 4
    N2V_WINDOW = 5
    N2V_EPOCHS = 5
    MIN_ENTITY_FREQ = 20
    MAX_CATEGORICAL_UNIQUE_FOR_GRAPH = 3000
else:
    N2V_DIM = 64
    N2V_WALK_LENGTH = 24
    N2V_NUM_WALKS = 8
    N2V_WINDOW = 8
    N2V_EPOCHS = 10
    MIN_ENTITY_FREQ = 10
    MAX_CATEGORICAL_UNIQUE_FOR_GRAPH = 8000

N2V_P = 1.0
N2V_Q = 1.0
N2V_WORKERS = max(os.cpu_count() - 1, 1) if os.cpu_count() else 1

N_FOLDS = 3
VAL_MONTHS = 9
TOP_KS = (0.01, 0.05, 0.10, 0.20)

LEAKAGE_COL_PATTERNS = [
    "motivo_encerramento", "encerramento", "dataencerramento", "data_encerramento",
    "resultado", "sentenca", "sentença", "condenacao", "condenação",
    "valor_condenacao", "valor_condenação", "valor_do_acordo", "acordo_valor",
    "target", "desfecho", "ganhou", "perdeu", "perda_realizada"
]

def save_report(df, filename):
    path = REPORTS_DIR / filename
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Relatório salvo em: {path}")
    return path


## 2. Funções utilitárias

In [ ]:
# ============================================================
# 2. Funções utilitárias
# ============================================================

def normalize_col_name(c):
    return re.sub(r"[^a-z0-9]+", "_", str(c).lower()).strip("_")

def find_first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    normalized_map = {normalize_col_name(c): c for c in df.columns}
    for cand in candidates:
        key = normalize_col_name(cand)
        if key in normalized_map:
            return normalized_map[key]
    return None

def is_leakage_col(col):
    c = normalize_col_name(col)
    return any(p in c for p in LEAKAGE_COL_PATTERNS)

def safe_auc(y_true, proba):
    try:
        if len(np.unique(y_true)) < 2:
            return np.nan
        return roc_auc_score(y_true, proba)
    except Exception:
        return np.nan

def safe_pr_auc(y_true, proba):
    try:
        if len(np.unique(y_true)) < 2:
            return np.nan
        return average_precision_score(y_true, proba)
    except Exception:
        return np.nan

def get_positive_proba(model, X):
    proba = model.predict_proba(X)
    if proba.ndim == 1:
        return proba
    return proba[:, 1]

def best_f1_threshold(y_true, proba):
    precision, recall, thresholds = precision_recall_curve(y_true, proba)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
    if len(thresholds) == 0:
        return 0.5, np.nan, np.nan, np.nan
    best_idx = int(np.nanargmax(f1[:-1]))
    return float(thresholds[best_idx]), float(precision[best_idx]), float(recall[best_idx]), float(f1[best_idx])

def metrics_at_threshold(y_true, proba, thr=0.5):
    pred = (proba >= thr).astype(int)
    return {
        "precision_perda": precision_score(y_true, pred, zero_division=0),
        "recall_perda": recall_score(y_true, pred, zero_division=0),
        "f1_perda": f1_score(y_true, pred, zero_division=0),
        "pred_perda_rate": float(pred.mean()),
    }

def topk_risco_perda_metrics(y_true, proba_perda, ks=TOP_KS):
    tmp = pd.DataFrame({"y_true": y_true, "proba_perda": proba_perda})
    tmp = tmp.sort_values("proba_perda", ascending=False).reset_index(drop=True)
    total_perdas = max(tmp["y_true"].sum(), 1)
    base_rate = tmp["y_true"].mean()
    rows = []
    for k in ks:
        n_top = max(1, int(np.ceil(len(tmp) * k)))
        top = tmp.head(n_top)
        precision_k = top["y_true"].mean()
        recall_k = top["y_true"].sum() / total_perdas
        lift_k = precision_k / base_rate if base_rate > 0 else np.nan
        rows.append({"top_k": k, "n_top": n_top, "precision_perda_at_k": precision_k, "recall_perda_at_k": recall_k, "lift_perda_at_k": lift_k, "taxa_perda_base": base_rate})
    return pd.DataFrame(rows)

def topk_prioridade_financeira_metrics(y_true, proba_perda, valor_ajuizado, ks=TOP_KS):
    tmp = pd.DataFrame({
        "y_true": y_true,
        "proba_perda": proba_perda,
        "valor_ajuizado": pd.Series(valor_ajuizado).fillna(0).clip(lower=0).values,
    })
    tmp["score_financeiro"] = tmp["proba_perda"] * tmp["valor_ajuizado"]
    tmp["valor_perda_real"] = tmp["y_true"] * tmp["valor_ajuizado"]
    tmp = tmp.sort_values("score_financeiro", ascending=False).reset_index(drop=True)
    total_valor = max(tmp["valor_ajuizado"].sum(), 1)
    total_valor_perdas = max(tmp["valor_perda_real"].sum(), 1)
    base_rate = tmp["y_true"].mean()
    rows = []
    for k in ks:
        n_top = max(1, int(np.ceil(len(tmp) * k)))
        top = tmp.head(n_top)
        precision_k = top["y_true"].mean()
        recall_k = top["y_true"].sum() / max(tmp["y_true"].sum(), 1)
        lift_k = precision_k / base_rate if base_rate > 0 else np.nan
        rows.append({
            "top_k": k,
            "n_top": n_top,
            "precision_perda_at_k": precision_k,
            "recall_perda_at_k": recall_k,
            "lift_perda_at_k": lift_k,
            "taxa_perda_base": base_rate,
            "valor_ajuizado_top": top["valor_ajuizado"].sum(),
            "share_valor_ajuizado_total": top["valor_ajuizado"].sum() / total_valor,
            "valor_ajuizado_perdas_top": top["valor_perda_real"].sum(),
            "share_valor_perdas_total": top["valor_perda_real"].sum() / total_valor_perdas,
        })
    return pd.DataFrame(rows)

def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True, min_frequency=20)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

def get_value_col(df):
    candidates = ["fe_valor_ajuizado", "Valorajuizado", "Valor_Ajuizado", "valor_ajuizado", "valor_causa", "Valor_Causa", "valor_da_causa"]
    return find_first_existing_col(df, candidates)


## 3. Carregamento dos dados

In [ ]:
# ============================================================
# 3. Carregar dados
# ============================================================

print("DEV_PATH:", DEV_PATH)
print("HOLDOUT_PATH:", HOLDOUT_PATH)
print("FEATURE_LIST_PATH:", FEATURE_LIST_PATH)

if not DEV_PATH.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {DEV_PATH}")
if not HOLDOUT_PATH.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {HOLDOUT_PATH}")

df_dev = pd.read_parquet(DEV_PATH)
df_holdout = pd.read_parquet(HOLDOUT_PATH)

print("df_dev:", df_dev.shape)
print("df_holdout:", df_holdout.shape)

ID_COL = find_first_existing_col(df_dev, ID_COL_CANDIDATES)
DATE_COL = find_first_existing_col(df_dev, DATE_COL_CANDIDATES)
VALUE_COL = get_value_col(df_dev)

if ID_COL is None:
    print("Nenhuma coluna de ID de processo encontrada. Criando process_id sintético.")
    df_dev = df_dev.reset_index(drop=True)
    df_holdout = df_holdout.reset_index(drop=True)
    df_dev["process_id_sintetico"] = "dev_" + df_dev.index.astype(str)
    df_holdout["process_id_sintetico"] = "holdout_" + df_holdout.index.astype(str)
    ID_COL = "process_id_sintetico"

if TARGET_COL not in df_dev.columns:
    raise ValueError(f"TARGET_COL='{TARGET_COL}' não encontrado no df_dev.")
if DATE_COL is None:
    raise ValueError("Nenhuma coluna de data encontrada. Ajuste DATE_COL_CANDIDATES.")

df_dev[DATE_COL] = pd.to_datetime(df_dev[DATE_COL], errors="coerce")
df_holdout[DATE_COL] = pd.to_datetime(df_holdout[DATE_COL], errors="coerce")

print("ID_COL:", ID_COL)
print("DATE_COL:", DATE_COL)
print("VALUE_COL:", VALUE_COL)
print("Taxa perda dev:", df_dev[TARGET_COL].mean())
print("Taxa perda holdout:", df_holdout[TARGET_COL].mean())


## 4. Seleção das features base e colunas para o grafo

As colunas do grafo são somente atributos disponíveis antes do desfecho. O notebook remove automaticamente colunas com nomes suspeitos de leakage.

In [ ]:
# ============================================================
# 4. Selecionar features base e colunas para grafo
# ============================================================

if FEATURE_LIST_PATH.exists():
    with open(FEATURE_LIST_PATH, "r", encoding="utf-8") as f:
        base_feature_list = [line.strip() for line in f if line.strip()]
    base_feature_list = [c for c in base_feature_list if c in df_dev.columns]
else:
    print("Feature list não encontrada. Usando seleção automática.")
    base_feature_list = [c for c in df_dev.columns if c not in [TARGET_COL, ID_COL] and c != DATE_COL and not is_leakage_col(c)]

base_feature_list = [
    c for c in base_feature_list
    if c in df_dev.columns and c in df_holdout.columns and c not in [TARGET_COL, ID_COL, DATE_COL] and not is_leakage_col(c)
]

preferred_graph_cols = [
    "Nm_Assunto", "Nm_Acao", "Nm_Produto", "Carteira", "UF", "Comarca",
    "Fasedoprocesso", "Fase", "Nm_Escritorio", "Escritorio", "Advogado",
    "Orgao_Julgador", "Vara", "Classe", "Tipo_Acao",
    "status_texto", "fase_processual_texto"
]

graph_cols = []
for c in preferred_graph_cols:
    real_c = find_first_existing_col(df_dev, [c])
    if real_c and real_c in df_holdout.columns and real_c not in graph_cols and not is_leakage_col(real_c):
        graph_cols.append(real_c)

for c in base_feature_list:
    if c in graph_cols or is_leakage_col(c):
        continue
    if str(df_dev[c].dtype) in ["object", "string", "category", "boolean"]:
        nunique = df_dev[c].nunique(dropna=True)
        if 2 <= nunique <= MAX_CATEGORICAL_UNIQUE_FOR_GRAPH:
            graph_cols.append(c)

graph_cols = [c for c in graph_cols if c in df_dev.columns and c in df_holdout.columns and not is_leakage_col(c)]

print("Qtd base_feature_list:", len(base_feature_list))
print("Qtd graph_cols:", len(graph_cols))
print(graph_cols[:80])


## 5. Construção de grafos e embeddings Node2Vec

O grafo é heterogêneo:

```text
P::<processo> -> C::<coluna>::<valor>
```

Também são criados nós de interação para capturar relações compostas, como `UF x Assunto`, `Comarca x Assunto` e `Produto x Assunto`, quando as colunas existem.

In [ ]:
# ============================================================
# 5. Node2Vec leve para grafo processo-entidade
# ============================================================

def clean_entity_value(x, max_len=80):
    if pd.isna(x):
        return None
    s = str(x).strip()
    if not s or s.lower() in ["nan", "none", "null", "nat"]:
        return None
    s = re.sub(r"\s+", " ", s)
    return s[:max_len]

def make_process_node(x):
    return f"P::{str(x)}"

def make_entity_node(col, val):
    return f"E::{normalize_col_name(col)}::{clean_entity_value(val)}"

def add_interaction_columns(df):
    df = df.copy()
    candidates = [
        ("UF", "Nm_Assunto"),
        ("Comarca", "Nm_Assunto"),
        ("Nm_Produto", "Nm_Assunto"),
        ("Carteira", "Nm_Assunto"),
        ("UF", "Nm_Acao"),
        ("Nm_Produto", "Nm_Acao"),
    ]
    for a, b in candidates:
        ca = find_first_existing_col(df, [a])
        cb = find_first_existing_col(df, [b])
        if ca and cb and ca in df.columns and cb in df.columns:
            new_col = f"graph_inter_{normalize_col_name(ca)}__{normalize_col_name(cb)}"
            va = df[ca].map(clean_entity_value)
            vb = df[cb].map(clean_entity_value)
            df[new_col] = np.where(va.notna() & vb.notna(), va.astype(str) + " | " + vb.astype(str), np.nan)
    return df

def build_bipartite_graph(df, id_col, graph_cols, min_entity_freq=MIN_ENTITY_FREQ):
    df_g = add_interaction_columns(df)
    graph_cols_ext = list(graph_cols) + [c for c in df_g.columns if c.startswith("graph_inter_")]

    entity_counter = Counter()
    for c in graph_cols_ext:
        if c not in df_g.columns or is_leakage_col(c):
            continue
        vals = df_g[c].map(clean_entity_value)
        for v in vals.dropna():
            entity_counter[(c, v)] += 1

    valid_entities = {(c, v) for (c, v), cnt in entity_counter.items() if cnt >= min_entity_freq}

    adjacency = defaultdict(set)
    process_nodes = []

    for _, row in df_g.iterrows():
        pnode = make_process_node(row[id_col])
        process_nodes.append(pnode)
        for c in graph_cols_ext:
            if c not in df_g.columns or is_leakage_col(c):
                continue
            v = clean_entity_value(row[c])
            if v is None or (c, v) not in valid_entities:
                continue
            enode = make_entity_node(c, v)
            adjacency[pnode].add(enode)
            adjacency[enode].add(pnode)

    adjacency = {k: sorted(v) for k, v in adjacency.items()}
    process_nodes = sorted(set(process_nodes))
    return adjacency, process_nodes, graph_cols_ext

def node2vec_next_node(adjacency, prev_node, current_node, p=1.0, q=1.0):
    neighbors = adjacency.get(current_node, [])
    if not neighbors:
        return None
    if prev_node is None:
        return random.choice(neighbors)
    prev_neighbors = set(adjacency.get(prev_node, []))
    weights = []
    for nb in neighbors:
        if nb == prev_node:
            weights.append(1.0 / p)
        elif nb in prev_neighbors:
            weights.append(1.0)
        else:
            weights.append(1.0 / q)
    weights = np.array(weights, dtype=float)
    weights = weights / weights.sum()
    return np.random.choice(neighbors, p=weights)

def generate_node2vec_walks(adjacency, start_nodes, walk_length=16, num_walks=4, p=1.0, q=1.0):
    walks = []
    nodes = list(start_nodes)
    for _ in range(num_walks):
        random.shuffle(nodes)
        for start in nodes:
            walk = [start]
            prev = None
            current = start
            for _step in range(walk_length - 1):
                nxt = node2vec_next_node(adjacency, prev, current, p=p, q=q)
                if nxt is None:
                    break
                walk.append(nxt)
                prev, current = current, nxt
            walks.append(walk)
    return walks

def fit_node2vec_embeddings(df, id_col, graph_cols, dim=N2V_DIM):
    adjacency, process_nodes, graph_cols_ext = build_bipartite_graph(df=df, id_col=id_col, graph_cols=graph_cols, min_entity_freq=MIN_ENTITY_FREQ)
    print(f"Grafo: {len(adjacency):,} nós | {sum(len(v) for v in adjacency.values())//2:,} arestas | {len(process_nodes):,} processos")

    if HAS_GENSIM:
        walks = generate_node2vec_walks(adjacency=adjacency, start_nodes=process_nodes, walk_length=N2V_WALK_LENGTH, num_walks=N2V_NUM_WALKS, p=N2V_P, q=N2V_Q)
        print(f"Walks geradas: {len(walks):,}")
        w2v = Word2Vec(sentences=walks, vector_size=dim, window=N2V_WINDOW, min_count=1, sg=1, workers=N2V_WORKERS, epochs=N2V_EPOCHS, seed=RANDOM_STATE)
        rows = []
        for pnode in process_nodes:
            raw_id = pnode.replace("P::", "", 1)
            vec = w2v.wv[pnode] if pnode in w2v.wv else np.zeros(dim)
            rows.append([raw_id] + list(vec))
        emb = pd.DataFrame(rows, columns=[id_col] + [f"graph_n2v_emb_{i:03d}" for i in range(dim)])
        return emb, {"method": "node2vec_gensim", "graph_cols_ext": graph_cols_ext}

    if HAS_SCIPY:
        print("Fallback: SVD em matriz processo-entidade.")
        process_nodes = sorted(process_nodes)
        process_idx = {p: i for i, p in enumerate(process_nodes)}
        entity_nodes = sorted([n for n in adjacency if not n.startswith("P::")])
        entity_idx = {e: i for i, e in enumerate(entity_nodes)}
        rows, cols, data = [], [], []
        for p in process_nodes:
            for e in adjacency.get(p, []):
                if e in entity_idx:
                    rows.append(process_idx[p]); cols.append(entity_idx[e]); data.append(1.0)
        X = sparse.csr_matrix((data, (rows, cols)), shape=(len(process_nodes), len(entity_nodes)))
        n_comp = min(dim, max(2, min(X.shape) - 1))
        svd = TruncatedSVD(n_components=n_comp, random_state=RANDOM_STATE)
        Z = svd.fit_transform(X)
        if n_comp < dim:
            Z = np.hstack([Z, np.zeros((Z.shape[0], dim - n_comp))])
        raw_ids = [p.replace("P::", "", 1) for p in process_nodes]
        emb = pd.DataFrame(Z, columns=[f"graph_n2v_emb_{i:03d}" for i in range(dim)])
        emb.insert(0, id_col, raw_ids)
        return emb, {"method": "svd_fallback", "graph_cols_ext": graph_cols_ext}

    raise ImportError("Nem gensim nem scipy/sklearn SVD estão disponíveis para gerar embeddings.")

def add_graph_embeddings(df_target, emb, id_col):
    out = df_target.copy()
    out[id_col] = out[id_col].astype(str)
    emb = emb.copy()
    emb[id_col] = emb[id_col].astype(str)
    out = out.merge(emb, on=id_col, how="left")
    graph_emb_cols = [c for c in out.columns if c.startswith("graph_n2v_emb_")]
    out[graph_emb_cols] = out[graph_emb_cols].fillna(0.0)
    return out, graph_emb_cols


## 6. Preprocessadores e modelos

Os modelos são equivalentes aos notebooks anteriores, agora com as features `graph_n2v_emb_*` adicionadas.

In [ ]:
# ============================================================
# 6. Preprocessadores e modelos
# ============================================================

def split_feature_types(df, feature_cols):
    numeric_features, categorical_features = [], []
    for c in feature_cols:
        if c not in df.columns:
            continue
        if pd.api.types.is_numeric_dtype(df[c]) or pd.api.types.is_bool_dtype(df[c]):
            numeric_features.append(c)
        elif pd.api.types.is_datetime64_any_dtype(df[c]):
            continue
        else:
            categorical_features.append(c)
    return numeric_features, categorical_features

def make_preprocessor(df, feature_cols, mode="onehot"):
    numeric_features, categorical_features = split_feature_types(df, feature_cols)
    numeric_pipe = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])
    if mode == "onehot":
        categorical_pipe = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", make_ohe())])
    elif mode == "ordinal":
        categorical_pipe = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))])
    else:
        raise ValueError("mode deve ser 'onehot' ou 'ordinal'.")
    preprocessor = ColumnTransformer(
        transformers=[("num", numeric_pipe, numeric_features), ("cat", categorical_pipe, categorical_features)],
        remainder="drop",
        sparse_threshold=0.3,
    )
    return preprocessor, numeric_features, categorical_features

def make_models(pos_rate=None):
    if pos_rate is None or pos_rate <= 0 or pos_rate >= 1:
        scale_pos_weight = 1.0
    else:
        scale_pos_weight = (1 - pos_rate) / pos_rate
    models = []
    models.append(("random_forest_onehot_graph", "onehot", RandomForestClassifier(n_estimators=300 if RUN_MODE == "full" else 160, min_samples_leaf=20, n_jobs=-1, class_weight="balanced_subsample", random_state=RANDOM_STATE)))
    models.append(("logistic_onehot_graph", "onehot", LogisticRegression(max_iter=1000, solver="saga", n_jobs=-1, class_weight="balanced", C=0.5, random_state=RANDOM_STATE)))
    models.append(("hgb_ordinal_graph", "ordinal", HistGradientBoostingClassifier(max_iter=220 if RUN_MODE == "full" else 120, learning_rate=0.05, max_leaf_nodes=31, l2_regularization=0.1, random_state=RANDOM_STATE)))
    if HAS_XGBOOST:
        models.append(("xgb_ordinal_graph", "ordinal", XGBClassifier(n_estimators=400 if RUN_MODE == "full" else 200, max_depth=4, learning_rate=0.045, subsample=0.85, colsample_bytree=0.85, min_child_weight=20, reg_lambda=3.0, objective="binary:logistic", eval_metric="logloss", n_jobs=-1, random_state=RANDOM_STATE, scale_pos_weight=scale_pos_weight)))
    if HAS_LIGHTGBM:
        models.append(("lgbm_ordinal_graph", "ordinal", LGBMClassifier(n_estimators=500 if RUN_MODE == "full" else 250, learning_rate=0.035, num_leaves=31, min_child_samples=60, subsample=0.85, colsample_bytree=0.85, reg_lambda=3.0, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)))
    return models

def build_pipeline(df_train, feature_cols, preprocess_mode, estimator):
    preprocessor, num_cols, cat_cols = make_preprocessor(df_train, feature_cols, mode=preprocess_mode)
    pipe = Pipeline(steps=[("preprocess", preprocessor), ("model", estimator)])
    return pipe, num_cols, cat_cols


## 7. Folds temporais walk-forward

In [ ]:
# ============================================================
# 7. Criar folds temporais
# ============================================================

def create_walk_forward_folds(df, date_col, n_folds=3, val_months=9):
    df_dates = df[[date_col]].dropna().copy()
    min_date = df_dates[date_col].min()
    max_date = df_dates[date_col].max()
    fold_rows = []
    val_end = max_date
    for fold in range(n_folds, 0, -1):
        val_end_i = val_end - pd.DateOffset(months=val_months * (n_folds - fold))
        val_start_i = val_end_i - pd.DateOffset(months=val_months) + pd.DateOffset(days=1)
        train_end_i = val_start_i - pd.DateOffset(days=1)
        if train_end_i <= min_date:
            continue
        train_mask = df[date_col] <= train_end_i
        val_mask = (df[date_col] >= val_start_i) & (df[date_col] <= val_end_i)
        if train_mask.sum() == 0 or val_mask.sum() == 0:
            continue
        fold_rows.append({"fold": fold, "train_start_date": min_date, "train_end_date": train_end_i, "val_start_date": val_start_i, "val_end_date": val_end_i, "qtd_train": int(train_mask.sum()), "qtd_val": int(val_mask.sum())})
    fold_rows = sorted(fold_rows, key=lambda x: x["fold"])
    for i, r in enumerate(fold_rows, 1):
        r["fold"] = i
    return fold_rows

folds = create_walk_forward_folds(df_dev, DATE_COL, n_folds=N_FOLDS, val_months=VAL_MONTHS)

fold_summary_rows = []
for f in folds:
    train_mask = df_dev[DATE_COL] <= f["train_end_date"]
    val_mask = (df_dev[DATE_COL] >= f["val_start_date"]) & (df_dev[DATE_COL] <= f["val_end_date"])
    y_train_tmp = df_dev.loc[train_mask, TARGET_COL]
    y_val_tmp = df_dev.loc[val_mask, TARGET_COL]
    fold_summary_rows.append({**f, "taxa_perda_train": y_train_tmp.mean(), "taxa_perda_val": y_val_tmp.mean(), "taxa_ganho_train": 1 - y_train_tmp.mean(), "taxa_ganho_val": 1 - y_val_tmp.mean()})

fold_summary = pd.DataFrame(fold_summary_rows)
save_report(fold_summary, "90_3e_graph_node2vec_folds_summary.csv")
fold_summary


## 8. Walk-forward com embeddings de grafo

Para cada fold:

1. Constrói o grafo com treino + validação usando apenas atributos disponíveis.
2. Gera embeddings Node2Vec.
3. Treina os modelos no treino.
4. Avalia na validação.
5. Mede PR-AUC, ROC-AUC, F1 e top-k de perda.

> Observação: este desenho é **label-free transductive** para os nós de validação/holdout, porque usa as conexões conhecidas do processo no momento da predição, mas não usa o target. Em produção, um processo novo também pode ser conectado ao grafo por suas entidades conhecidas.

In [ ]:
# ============================================================
# 8. Walk-forward
# ============================================================

all_model_results = []
all_topk_results = []

for fold in folds:
    fold_id = fold["fold"]
    print("\n" + "="*100)
    print(f"Fold {fold_id}")
    print("="*100)

    train_mask = df_dev[DATE_COL] <= fold["train_end_date"]
    val_mask = (df_dev[DATE_COL] >= fold["val_start_date"]) & (df_dev[DATE_COL] <= fold["val_end_date"])

    df_train_raw = df_dev.loc[train_mask].copy()
    df_val_raw = df_dev.loc[val_mask].copy()

    graph_universe = pd.concat([df_train_raw.drop(columns=[TARGET_COL], errors="ignore"), df_val_raw.drop(columns=[TARGET_COL], errors="ignore")], axis=0, ignore_index=True)
    emb_fold, emb_meta = fit_node2vec_embeddings(graph_universe, id_col=ID_COL, graph_cols=graph_cols, dim=N2V_DIM)

    df_train, graph_emb_cols = add_graph_embeddings(df_train_raw, emb_fold, ID_COL)
    df_val, _ = add_graph_embeddings(df_val_raw, emb_fold, ID_COL)

    feature_cols = [c for c in base_feature_list if c in df_train.columns and c in df_val.columns and not is_leakage_col(c)] + graph_emb_cols
    feature_cols = list(dict.fromkeys(feature_cols))

    X_train = df_train[feature_cols].copy()
    y_train = df_train[TARGET_COL].astype(int).copy()
    X_val = df_val[feature_cols].copy()
    y_val = df_val[TARGET_COL].astype(int).copy()

    print(f"Treino: {X_train.shape} | Val: {X_val.shape} | perda_train={y_train.mean():.4f} | perda_val={y_val.mean():.4f}")
    print(f"Embeddings: {len(graph_emb_cols)} | Método: {emb_meta['method']}")

    for model_name, preprocess_mode, estimator in make_models(pos_rate=y_train.mean()):
        print(f"Treinando {model_name} | preprocess={preprocess_mode}")
        t0 = time.time()
        pipe, num_cols, cat_cols = build_pipeline(X_train, feature_cols, preprocess_mode, estimator)
        pipe.fit(X_train, y_train)
        proba_val = get_positive_proba(pipe, X_val)

        roc_auc = safe_auc(y_val, proba_val)
        pr_auc = safe_pr_auc(y_val, proba_val)
        best_thr, best_p, best_r, best_f1 = best_f1_threshold(y_val, proba_val)
        thr_05 = metrics_at_threshold(y_val, proba_val, thr=0.5)

        all_model_results.append({
            "fold": fold_id,
            "model": model_name,
            "preprocess_mode": preprocess_mode,
            "roc_auc_perda": roc_auc,
            "pr_auc_perda": pr_auc,
            "taxa_perda_train": y_train.mean(),
            "taxa_perda_val": y_val.mean(),
            "best_f1_threshold_perda": best_thr,
            "best_f1_precision_perda": best_p,
            "best_f1_recall_perda": best_r,
            "best_f1_perda": best_f1,
            "precision_perda_t05": thr_05["precision_perda"],
            "recall_perda_t05": thr_05["recall_perda"],
            "f1_perda_t05": thr_05["f1_perda"],
            "pred_perda_rate_t05": thr_05["pred_perda_rate"],
            "qtd_train": len(df_train),
            "qtd_val": len(df_val),
            "qtd_features": len(feature_cols),
            "qtd_graph_embeddings": len(graph_emb_cols),
            "graph_embedding_method": emb_meta["method"],
            "runtime_sec": time.time() - t0,
        })

        topk = topk_risco_perda_metrics(y_val.values, proba_val, ks=TOP_KS)
        topk["fold"] = fold_id
        topk["model"] = model_name
        all_topk_results.append(topk)

model_results = pd.DataFrame(all_model_results)
topk_results = pd.concat(all_topk_results, ignore_index=True) if all_topk_results else pd.DataFrame()

save_report(model_results, "91_3e_graph_node2vec_walk_forward_results.csv")
save_report(topk_results, "92_3e_graph_node2vec_walk_forward_topk.csv")

model_results.sort_values(["pr_auc_perda", "roc_auc_perda"], ascending=False).head(20)


## 9. Resumo por modelo

In [ ]:
# ============================================================
# 9. Resumo por modelo
# ============================================================

model_summary = (
    model_results
    .groupby(["model", "preprocess_mode"])
    .agg(
        mean_roc_auc_perda=("roc_auc_perda", "mean"),
        std_roc_auc_perda=("roc_auc_perda", "std"),
        mean_pr_auc_perda=("pr_auc_perda", "mean"),
        std_pr_auc_perda=("pr_auc_perda", "std"),
        mean_taxa_perda_val=("taxa_perda_val", "mean"),
        mean_best_f1_perda=("best_f1_perda", "mean"),
        mean_best_f1_threshold_perda=("best_f1_threshold_perda", "mean"),
        mean_precision_perda_t05=("precision_perda_t05", "mean"),
        mean_recall_perda_t05=("recall_perda_t05", "mean"),
        mean_f1_perda_t05=("f1_perda_t05", "mean"),
        mean_pred_perda_rate_t05=("pred_perda_rate_t05", "mean"),
        mean_qtd_graph_embeddings=("qtd_graph_embeddings", "mean"),
    )
    .reset_index()
    .sort_values("mean_pr_auc_perda", ascending=False)
)

save_report(model_summary, "93_3e_graph_node2vec_model_summary.csv")
model_summary


## 10. Top-k médio por modelo

In [ ]:
# ============================================================
# 10. Resumo top-k por modelo
# ============================================================

topk_summary = (
    topk_results
    .groupby(["model", "top_k"])
    .agg(
        mean_precision_perda_at_k=("precision_perda_at_k", "mean"),
        mean_recall_perda_at_k=("recall_perda_at_k", "mean"),
        mean_lift_perda_at_k=("lift_perda_at_k", "mean"),
        mean_taxa_perda_base=("taxa_perda_base", "mean"),
    )
    .reset_index()
    .sort_values(["top_k", "mean_precision_perda_at_k"], ascending=[True, False])
)

save_report(topk_summary, "94_3e_graph_node2vec_topk_summary.csv")
topk_summary


## 11. Treinar melhor modelo no DEV completo e avaliar no holdout temporal

Nesta etapa, o grafo é construído com `DEV + HOLDOUT`, mas sem labels e sem colunas de leakage. Isso simula o cenário de produção em que os processos do período atual já possuem entidades conhecidas e podem ser conectados ao grafo.

In [ ]:
# ============================================================
# 11. Treino final no DEV completo + avaliação no holdout
# ============================================================

best_row = model_summary.iloc[0]
best_model_name = best_row["model"]
best_preprocess_mode = best_row["preprocess_mode"]

print("Melhor modelo por PR-AUC perda no WF:", best_model_name)
print("Preprocessamento:", best_preprocess_mode)

graph_universe_final = pd.concat([df_dev.drop(columns=[TARGET_COL], errors="ignore"), df_holdout.drop(columns=[TARGET_COL], errors="ignore")], axis=0, ignore_index=True)
emb_final, emb_meta_final = fit_node2vec_embeddings(graph_universe_final, id_col=ID_COL, graph_cols=graph_cols, dim=N2V_DIM)

df_dev_model, graph_emb_cols = add_graph_embeddings(df_dev, emb_final, ID_COL)
df_holdout_model, _ = add_graph_embeddings(df_holdout, emb_final, ID_COL)

selected_features = [c for c in base_feature_list if c in df_dev_model.columns and c in df_holdout_model.columns and not is_leakage_col(c)] + graph_emb_cols
selected_features = list(dict.fromkeys(selected_features))

X_dev = df_dev_model[selected_features].copy()
y_dev = df_dev_model[TARGET_COL].astype(int).copy()
X_holdout = df_holdout_model[selected_features].copy()
y_holdout = df_holdout_model[TARGET_COL].astype(int).copy()

candidate_models = make_models(pos_rate=y_dev.mean())
selected_tuple = [m for m in candidate_models if m[0] == best_model_name]
if not selected_tuple:
    raise ValueError(f"Modelo {best_model_name} não encontrado em make_models().")
_, preprocess_mode, estimator = selected_tuple[0]

best_pipeline, num_cols, cat_cols = build_pipeline(X_dev, selected_features, preprocess_mode, estimator)
best_pipeline.fit(X_dev, y_dev)

proba_perda_holdout = get_positive_proba(best_pipeline, X_holdout)

holdout_roc_auc_perda = safe_auc(y_holdout, proba_perda_holdout)
holdout_pr_auc_perda = safe_pr_auc(y_holdout, proba_perda_holdout)
best_thr, best_p, best_r, best_f1 = best_f1_threshold(y_holdout, proba_perda_holdout)
thr_05 = metrics_at_threshold(y_holdout, proba_perda_holdout, thr=0.5)

holdout_metrics = pd.DataFrame([{
    "model": best_model_name,
    "preprocess_mode": preprocess_mode,
    "roc_auc_perda": holdout_roc_auc_perda,
    "pr_auc_perda": holdout_pr_auc_perda,
    "taxa_perda_holdout": y_holdout.mean(),
    "taxa_ganho_holdout": 1 - y_holdout.mean(),
    "best_f1_threshold_perda": best_thr,
    "best_f1_precision_perda": best_p,
    "best_f1_recall_perda": best_r,
    "best_f1_perda": best_f1,
    "precision_perda_t05": thr_05["precision_perda"],
    "recall_perda_t05": thr_05["recall_perda"],
    "f1_perda_t05": thr_05["f1_perda"],
    "pred_perda_rate_t05": thr_05["pred_perda_rate"],
    "qtd_dev": len(df_dev_model),
    "qtd_holdout": len(df_holdout_model),
    "qtd_features": len(selected_features),
    "qtd_graph_embeddings": len(graph_emb_cols),
    "graph_embedding_method": emb_meta_final["method"],
}])

save_report(holdout_metrics, "95_3e_graph_node2vec_holdout_best_model_metrics.csv")
holdout_metrics


## 12. Holdout top-k — risco puro de perda

In [ ]:
# ============================================================
# 12. Holdout top-k risco puro
# ============================================================

holdout_topk_perda = topk_risco_perda_metrics(y_true=y_holdout.values, proba_perda=proba_perda_holdout, ks=TOP_KS)
holdout_topk_perda["model"] = best_model_name
holdout_topk_perda["preprocess_mode"] = preprocess_mode

save_report(holdout_topk_perda, "96_3e_graph_node2vec_holdout_topk_risco_perda.csv")
holdout_topk_perda


## 13. Holdout top-k — prioridade financeira

In [ ]:
# ============================================================
# 13. Holdout top-k prioridade financeira
# ============================================================

VALUE_COL = get_value_col(df_holdout_model)
print("VALUE_COL:", VALUE_COL)

if VALUE_COL and VALUE_COL in df_holdout_model.columns:
    holdout_topk_financeiro = topk_prioridade_financeira_metrics(y_true=y_holdout.values, proba_perda=proba_perda_holdout, valor_ajuizado=df_holdout_model[VALUE_COL], ks=TOP_KS)
    holdout_topk_financeiro["model"] = best_model_name
    holdout_topk_financeiro["preprocess_mode"] = preprocess_mode
    save_report(holdout_topk_financeiro, "97_3e_graph_node2vec_holdout_topk_prioridade_financeira.csv")
else:
    print("Nenhuma coluna de valor encontrada. Pulando prioridade financeira.")
    holdout_topk_financeiro = pd.DataFrame()

holdout_topk_financeiro


## 14. Ranking do holdout para consumo jurídico

Gera um arquivo com score de perda, ranking de risco e ranking financeiro.

In [ ]:
# ============================================================
# 14. Ranking para consumo jurídico
# ============================================================

ranking_cols = []
for c in [ID_COL, DATE_COL, VALUE_COL, "Nm_Assunto", "Nm_Acao", "Nm_Produto", "Carteira", "UF", "Comarca", "Fasedoprocesso", "Nm_Escritorio", "Advogado"]:
    if c and c in df_holdout_model.columns and c not in ranking_cols:
        ranking_cols.append(c)

ranking_holdout = df_holdout_model[ranking_cols].copy() if ranking_cols else pd.DataFrame(index=df_holdout_model.index)
ranking_holdout[TARGET_COL] = y_holdout.values
ranking_holdout["target_semantica"] = np.where(ranking_holdout[TARGET_COL] == 1, "banco_perdeu", "banco_ganhou")
ranking_holdout["proba_perda"] = proba_perda_holdout
ranking_holdout["score_risco_perda"] = proba_perda_holdout
ranking_holdout["ranking_risco_perda"] = ranking_holdout["score_risco_perda"].rank(method="first", ascending=False).astype(int)

if VALUE_COL and VALUE_COL in df_holdout_model.columns:
    ranking_holdout["valor_ajuizado"] = pd.to_numeric(df_holdout_model[VALUE_COL], errors="coerce").fillna(0).clip(lower=0)
    ranking_holdout["score_prioridade_financeira"] = ranking_holdout["proba_perda"] * ranking_holdout["valor_ajuizado"]
    ranking_holdout["ranking_prioridade_financeira"] = ranking_holdout["score_prioridade_financeira"].rank(method="first", ascending=False).astype(int)

ranking_holdout["model"] = best_model_name
ranking_holdout["preprocess_mode"] = preprocess_mode
ranking_holdout["graph_embedding_method"] = emb_meta_final["method"]

ranking_path = PROCESSED_DIR / "ranking_holdout_risco_perda_graph_node2vec_3e.parquet"
ranking_holdout.to_parquet(ranking_path, index=False)

save_report(ranking_holdout.head(1000), "98_3e_graph_node2vec_ranking_holdout_preview.csv")
print("Ranking salvo em:", ranking_path)
ranking_holdout.head(20)


## 15. Matriz de confusão e relatório de classificação

O threshold 0.5 é apenas referência. Para uso jurídico, a decisão principal deve vir por **ranking**, **top-k** e, depois, por probabilidade calibrada.

In [ ]:
# ============================================================
# 15. Classification report e confusion matrix
# ============================================================

pred_perda_holdout_t05 = (proba_perda_holdout >= 0.5).astype(int)

print("Classification report — threshold 0.5")
print("Classe 0 = banco ganhou")
print("Classe 1 = banco perdeu")
print(classification_report(y_holdout, pred_perda_holdout_t05, target_names=["banco_ganhou", "banco_perdeu"], zero_division=0))

print("Confusion matrix — threshold 0.5")
print("Linhas = real | Colunas = previsto")
print(confusion_matrix(y_holdout, pred_perda_holdout_t05))


## 16. Importância das features

In [ ]:
# ============================================================
# 16. Feature importance quando disponível
# ============================================================

def get_feature_names_from_preprocessor(pipe, input_features):
    pre = pipe.named_steps["preprocess"]
    try:
        return pre.get_feature_names_out()
    except Exception:
        return np.array(input_features)

def extract_feature_importance(pipe, input_features):
    model = pipe.named_steps["model"]
    if hasattr(model, "feature_importances_"):
        names = get_feature_names_from_preprocessor(pipe, input_features)
        imp = model.feature_importances_
        n = min(len(names), len(imp))
        return pd.DataFrame({"feature": names[:n], "importance": imp[:n]}).sort_values("importance", ascending=False)
    if hasattr(model, "coef_"):
        names = get_feature_names_from_preprocessor(pipe, input_features)
        coef = np.ravel(model.coef_)
        n = min(len(names), len(coef))
        return pd.DataFrame({"feature": names[:n], "importance": np.abs(coef[:n]), "coef": coef[:n]}).sort_values("importance", ascending=False)
    return pd.DataFrame()

feature_importance = extract_feature_importance(best_pipeline, selected_features)
if not feature_importance.empty:
    save_report(feature_importance, "99_3e_graph_node2vec_feature_importance.csv")
    display(feature_importance.head(50))
else:
    print("Modelo selecionado não expõe feature_importances_ ou coef_.")


## 17. Comparação opcional com resultados anteriores

Se os CSVs dos notebooks anteriores existirem, esta célula compara o notebook 3E com os resultados de 3A/3B/3C/3D.

In [ ]:
# ============================================================
# 17. Comparação opcional com notebooks anteriores
# ============================================================

comparison_rows = []

def add_if_exists(label, filename):
    path = REPORTS_DIR / filename
    if path.exists():
        df = pd.read_csv(path)
        if df.empty:
            return
        row = df.iloc[0].to_dict()
        comparison_rows.append({
            "notebook": label,
            "model": row.get("model", label),
            "holdout_pr_auc_perda": row.get("pr_auc_perda", row.get("holdout_pr_auc_perda", np.nan)),
            "holdout_roc_auc_perda": row.get("roc_auc_perda", row.get("holdout_roc_auc_perda", np.nan)),
            "top5_precision_perda": np.nan,
            "top10_fin_share_valor_perdas": np.nan,
        })

add_if_exists("3A_baseline", "27_holdout_best_baseline_metrics_corrigido.csv")
add_if_exists("3B_encoders", "42_3b_holdout_best_model_metrics.csv")
add_if_exists("3C_lgbm_xgb", "75_3c_lgbm_xgb_holdout_best_model_metrics.csv")
add_if_exists("3D_text", "76_3d_text_holdout_best_model_metrics.csv")

top5 = holdout_topk_perda.loc[holdout_topk_perda["top_k"] == 0.05, "precision_perda_at_k"]
top10fin = pd.Series(dtype=float)
if not holdout_topk_financeiro.empty:
    top10fin = holdout_topk_financeiro.loc[holdout_topk_financeiro["top_k"] == 0.10, "share_valor_perdas_total"]

comparison_rows.append({
    "notebook": "3E_graph_node2vec",
    "model": best_model_name,
    "holdout_pr_auc_perda": holdout_pr_auc_perda,
    "holdout_roc_auc_perda": holdout_roc_auc_perda,
    "top5_precision_perda": float(top5.iloc[0]) if len(top5) else np.nan,
    "top10_fin_share_valor_perdas": float(top10fin.iloc[0]) if len(top10fin) else np.nan,
})

comparison = pd.DataFrame(comparison_rows)
if not comparison.empty:
    save_report(comparison, "100_3e_graph_node2vec_comparison_previous_steps.csv")
comparison


## 18. Summary final

In [ ]:
# ============================================================
# 18. Summary final
# ============================================================

summary_step_3e = pd.DataFrame([{
    "target_semantica": "0=banco_ganhou | 1=banco_perdeu",
    "notebook": "07_03E_graph_node2vec_features_early_stage",
    "run_mode": RUN_MODE,
    "best_model_3e_walk_forward": best_model_name,
    "best_model_3e_preprocess": preprocess_mode,
    "best_model_3e_mean_pr_auc_perda_wf": float(best_row["mean_pr_auc_perda"]),
    "best_model_3e_mean_roc_auc_perda_wf": float(best_row["mean_roc_auc_perda"]),
    "holdout_pr_auc_perda": float(holdout_pr_auc_perda),
    "holdout_roc_auc_perda": float(holdout_roc_auc_perda),
    "taxa_perda_holdout": float(y_holdout.mean()),
    "taxa_ganho_holdout": float(1 - y_holdout.mean()),
    "qtd_features": len(selected_features),
    "qtd_graph_embeddings": len(graph_emb_cols),
    "graph_embedding_method": emb_meta_final["method"],
    "n2v_dim": N2V_DIM,
    "n2v_walk_length": N2V_WALK_LENGTH,
    "n2v_num_walks": N2V_NUM_WALKS,
    "n2v_window": N2V_WINDOW,
    "n2v_epochs": N2V_EPOCHS,
    "min_entity_freq": MIN_ENTITY_FREQ,
    "qtd_dev": len(df_dev_model),
    "qtd_holdout": len(df_holdout_model),
    "ranking_risco_holdout_path": str(ranking_path),
}])

save_report(summary_step_3e, "101_summary_step_3e_graph_node2vec.csv")
summary_step_3e.T


## O que verificar após rodar

1. Se `holdout_pr_auc_perda` melhorou em relação ao melhor notebook anterior.
2. Se `precision_perda_at_k` em top-1%, top-5% e top-10% melhorou.
3. Se `share_valor_perdas_total` no top-10% financeiro melhorou.
4. Se as features `graph_n2v_emb_*` aparecem com alguma importância relevante.
5. Se o ganho do grafo é estável nos folds ou aparece apenas no holdout.
6. Se há risco de leakage indireto em alguma coluna usada como nó do grafo.

Se o ganho for pequeno, a decisão correta pode ser manter o grafo como módulo experimental e seguir para calibração/probabilidade final com o melhor pipeline já validado.